In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
patientpayer_table = dbutils.widgets.get("patientpayer_table")
patient_table = dbutils.widgets.get("patient_table")
branch_table_cubhub = dbutils.widgets.get("branch_table_cubhub")
payer_table = dbutils.widgets.get("payer_table")
financialclass_table = dbutils.widgets.get("financialclass_table")
payment_table = dbutils.widgets.get("payment_table")
paymentline_table = dbutils.widgets.get("paymentline_table")
writeoff_table = dbutils.widgets.get("writeoff_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW transactions_src AS
SELECT 
    try_CAST(ReportingDate AS DATE) AS ReportingDate,
    CAST(PaymentID AS BIGINT) AS PaymentID,
    CAST(FacilityCode AS INT) AS FacilityCode,
    CAST(AcctNbr AS STRING) AS AcctNbr,
    CAST(TransDate AS INT) AS TransDate,
    CAST(EntryDate AS INT) AS EntryDate,
    CAST(TransAmt AS DOUBLE) AS TransAmt,
    CAST(TransCode AS STRING) AS TransCode,
    CAST(TransDesc AS STRING) AS TransDesc,
    CAST(TransType AS STRING) AS TransType,
    NULL AS InsCode,
    NULL AS Payor,
    NULL AS FinClass,
    NULL AS Coinsurance,
    NULL AS Deductible,
    NULL AS CoPay,
    NULL AS PatResp,
    NULL AS BatchID,
    CAST(SourceSystemKey AS INT) AS SourceSystemKey
FROM (
  WITH 
  transactions_cte AS (
    -- Payments WITH paymentline (line-level amounts)
    SELECT
        CAST('{fetch_date}' AS DATE) AS ReportingDate,
        pl.Id AS PaymentID,
        b.ExternalId AS FacilityCode,
        bl.ClaimNumber AS AcctNbr,
        DATE_FORMAT(pay.PaymentDate, 'yyyyMMdd') AS TransDate,
        DATE_FORMAT(pay.CreatedDate, 'yyyyMMdd') AS EntryDate,
        pl.Amount AS TransAmt,
        CASE pay.PaymentType
            WHEN 1 THEN 'Credit Card'
            WHEN 2 THEN 'EFT'
            WHEN 3 THEN 'Manual Adj'
            WHEN 5 THEN 'Refund'
            WHEN 7 THEN 'Transfer In'
            WHEN 8 THEN 'Transfer Out'
            WHEN 9 THEN 'Reversal'
            WHEN 10 THEN 'Credit Memo'
            WHEN 11 THEN 'CC Reversal'
            WHEN 12 THEN 'ACH'
            WHEN 13 THEN 'Write-On'
            ELSE CAST(pay.PaymentType AS STRING)
        END AS TransCode,
        pay.PaymentDetails AS TransDesc,
        CASE pay.PaymentType
            WHEN 1 THEN 'Payment'
            WHEN 2 THEN 'Payment'
            WHEN 12 THEN 'Payment'
            ELSE 'Adjustment'
        END AS TransType,
        NULL AS InsCode,
        NULL AS Payor,
        fc.Abbreviation AS FinClass,
        NULL AS Coinsurance,
        NULL AS Deductible,
        NULL AS CoPay,
        NULL AS PatResp,
        NULL AS BatchID,
        '19' AS SourceSystemKey
    FROM {source_table} bl
    JOIN {patientpayer_table} pp ON pp.Id = bl.PatientPayerId
    JOIN {patient_table} p ON p.Id = pp.PatientId
    JOIN {branch_table_cubhub} b ON b.Id = p.BranchId
    JOIN {payer_table} payer ON payer.Id = pp.PayerId
    LEFT JOIN {financialclass_table} fc ON fc.Id = payer.FinancialClassId
    JOIN {payment_table} pay ON bl.Id = pay.BillingLedgerId
    JOIN {paymentline_table} pl ON pl.PaymentId = pay.Id
    WHERE bl.isActive='true'

    UNION ALL

    -- Payments WITHOUT paymentline (payment-level amounts)
    SELECT
        CAST('{fetch_date}' AS DATE) AS ReportingDate,
        pay.Id AS PaymentID,
        b.ExternalId AS FacilityCode,
        bl.ClaimNumber AS AcctNbr,
        DATE_FORMAT(pay.PaymentDate, 'yyyyMMdd') AS TransDate,
        DATE_FORMAT(pay.CreatedDate, 'yyyyMMdd') AS EntryDate,
        pay.Amount AS TransAmt,
        CASE pay.PaymentType
            WHEN 1 THEN 'Credit Card'
            WHEN 2 THEN 'EFT'
            WHEN 3 THEN 'Manual Adj'
            WHEN 5 THEN 'Refund'
            WHEN 7 THEN 'Transfer In'
            WHEN 8 THEN 'Transfer Out'
            WHEN 9 THEN 'Reversal'
            WHEN 10 THEN 'Credit Memo'
            WHEN 11 THEN 'CC Reversal'
            WHEN 12 THEN 'ACH'
            WHEN 13 THEN 'Write-On'
            ELSE CAST(pay.PaymentType AS STRING)
        END AS TransCode,
        pay.PaymentDetails AS TransDesc,
        CASE pay.PaymentType
            WHEN 1 THEN 'Payment'
            WHEN 2 THEN 'Payment'
            WHEN 12 THEN 'Payment'
            ELSE 'Adjustment'
        END AS TransType,
        NULL AS InsCode,
        NULL AS Payor,
        fc.Abbreviation AS FinClass,
        NULL AS Coinsurance,
        NULL AS Deductible,
        NULL AS CoPay,
        NULL AS PatResp,
        NULL AS BatchID,
        '19' AS SourceSystemKey
    FROM {source_table} bl
    JOIN {patientpayer_table} pp ON pp.Id = bl.PatientPayerId
    JOIN {patient_table} p ON p.Id = pp.PatientId
    JOIN {branch_table_cubhub} b ON b.Id = p.BranchId
    JOIN {payer_table} payer ON payer.Id = pp.PayerId
    LEFT JOIN {financialclass_table} fc ON fc.Id = payer.FinancialClassId
    JOIN {payment_table} pay ON bl.Id = pay.BillingLedgerId
    WHERE NOT EXISTS (
        SELECT 1 FROM {paymentline_table} pl WHERE pl.PaymentId = pay.Id
    )
    -- AND bl.claimNumber = "288928FI1240"
    AND bl.isActive='true'
    UNION

    -- Adjustments from writeoff table
    SELECT
        CAST('{fetch_date}' AS DATE) AS ReportingDate,
        wo.Id AS PaymentID,
        b.ExternalId AS FacilityCode,
        bl.ClaimNumber AS AcctNbr,
        DATE_FORMAT(wo.DateBilled, 'yyyyMMdd') AS TransDate,
        DATE_FORMAT(wo.CreatedDate, 'yyyyMMdd') AS EntryDate,
        wo.Amount AS TransAmt,
        'Credit Memo' AS TransCode,
        wo.Notes AS TransDesc,
        'Adjustment' AS TransType,
        NULL AS InsCode,
        NULL AS Payor,
        fc.Abbreviation AS FinClass,
        NULL AS Coinsurance,
        NULL AS Deductible,
        NULL AS CoPay,
        NULL AS PatResp,
        NULL AS BatchID,
        '19' AS SourceSystemKey
      FROM {source_table} bl
      JOIN {patientpayer_table} pp ON pp.Id = bl.PatientPayerId
      JOIN {patient_table} p ON p.Id = pp.PatientId
      JOIN {branch_table_cubhub} b ON b.Id = p.BranchId
      JOIN {payer_table} payer ON payer.Id = pp.PayerId
      LEFT JOIN {financialclass_table} fc ON fc.Id = payer.FinancialClassId
      JOIN {writeoff_table} wo ON bl.Id = wo.BillingLedgerId
      AND bl.claimNumber = "307907FI1938"
      AND bl.isActive='true'
  ),
  transactions_clean AS (
    SELECT *,
    row_number() OVER (PARTITION BY AcctNbr ORDER BY AcctNbr ) AS rn
    FROM transactions_cte
  )
  SELECT ReportingDate, PaymentID, FacilityCode, AcctNbr, TransDate, EntryDate, TransAmt, TransCode, TransDesc, TransType, InsCode, Payor, FinClass, Coinsurance, Deductible, CoPay, PatResp, BatchID, SourceSystemKey
  FROM transactions_clean
  WHERE rn=1
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING transactions_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 19

WHEN MATCHED THEN
UPDATE SET
    tgt.PaymentID = src.PaymentID,
    tgt.FacilityCode = src.FacilityCode,
    tgt.TransDate = src.TransDate,
    tgt.EntryDate = src.EntryDate,
    tgt.TransAmt = src.TransAmt,
    tgt.TransCode = src.TransCode,
    tgt.TransDesc = src.TransDesc,
    tgt.TransType = src.TransType,
    tgt.InsCode = src.InsCode,
    tgt.Payor = src.Payor,
    tgt.FinClass = src.FinClass,
    tgt.Coinsurance = src.Coinsurance,
    tgt.Deductible = src.Deductible,
    tgt.CoPay = src.CoPay,
    tgt.PatResp = src.PatResp,
    tgt.BatchID = src.BatchID,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    PaymentID,
    FacilityCode,
    AcctNbr,
    TransDate,
    EntryDate,
    TransAmt,
    TransCode,
    TransDesc,
    TransType,
    InsCode,
    Payor,
    FinClass,
    Coinsurance,
    Deductible,
    CoPay,
    PatResp,
    BatchID,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.PaymentID,
    src.FacilityCode,
    src.AcctNbr,
    src.TransDate,
    src.EntryDate,
    src.TransAmt,
    src.TransCode,
    src.TransDesc,
    src.TransType,
    src.InsCode,
    src.Payor,
    src.FinClass,
    src.Coinsurance,
    src.Deductible,
    src.CoPay,
    src.PatResp,
    src.BatchID,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)